# Introduction to Statistical Inference

**STA 504 2.0 Statistical Inference** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta504.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Description and generalization](#theory-01)
- [Parameters, statistics, estimators and estimates](#theory-02)
- [Parametric and nonparametric approaches](#theory-03)
- [Point and interval estimation](#theory-04)
- [Sampling distributions and standard errors](#theory-05)
- [Estimator properties](#theory-06)
- [Estimation methods](#theory-07)
- [Reporting and limitations](#theory-08)

## Detailed theory · Foundations of statistical inference

<a id="theory-01"></a>

### Description and generalization
A descriptive calculation summarizes observed cases. Inference concerns an unobserved population quantity under a sampling or probability model. Define the target, unit, selection mechanism and period before generalizing. Random sampling and independence are substantive conditions, not properties created by a formula.

<a id="theory-02"></a>

### Parameters, statistics, estimators and estimates
A parameter such as $\mu$, $\sigma^2$ or p describes the population/model. A statistic is a function of the sample without unknown parameters. An estimator is a statistic used to estimate a parameter; an estimate is its observed number. The same symbol can be used carelessly for a rule and a value, so distinguish random $\bar X$ from observed $\bar x$.

<a id="theory-03"></a>

### Parametric and nonparametric approaches
Parametric inference specifies a distribution family indexed by parameters. Nonparametric inference relaxes parts of that form while retaining assumptions about sampling and the target. Choosing between them is about the question and the model, not a rule that small datasets must be nonparametric.

<a id="theory-04"></a>

### Point and interval estimation
A point estimate supplies one value; an interval estimate communicates uncertainty using a procedure with stated coverage properties. An estimator can be precise under its model yet biased for the actual target because of frame or measurement problems.

<a id="theory-05"></a>

### Sampling distributions and standard errors
An estimator varies across repeated samples. Its sampling distribution is different from the distribution of individual observations. SE is its sampling SD, estimated when parameters are unknown. For independent identical observations with finite variance, $Var(\bar X)=\sigma^2/n$. For Bernoulli observations, $Var(\hat p)=p(1-p)/n$.

<a id="theory-06"></a>

### Estimator properties
$$Bias(\hat\theta)=E(\hat\theta)-\theta,\quad
MSE(\hat\theta)=Var(\hat\theta)+Bias(\hat\theta)^2.$$
Unbiasedness is a fixed-sample expectation property. Consistency means the estimator approaches the target in probability as information grows. Efficiency compares variances or error risk within a specified class/model. A biased estimator can have lower MSE than an unbiased one.

<a id="theory-07"></a>

### Estimation methods
Maximum likelihood maximizes the probability model's likelihood for the observed sample. Method of moments matches sample and theoretical moments. Least squares minimizes squared discrepancies in a specified mean model. They can agree in special cases but are different principles; existence, support and identifiability must be checked.

<a id="theory-08"></a>

### Reporting and limitations
State the target parameter, estimator, model, data design, estimate, SE/interval and potential biases. Software output is a calculation under assumptions. A standard error does not automatically include missing-data bias, model misspecification or the uncertainty of choices made after seeing the results.

## Concepts covered

**Course points:** Point estimators; sampling distributions; bias/variability; standard error; CLT; interval estimation.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Estimator properties: expectation, bias and MSE table

### Question and assumptions

An illustrative estimator of a parameter $\theta=2$ takes values 0,2,4 with probabilities .25,.5,.25.

### Procedure

1. Calculate probability-weighted estimator values.
2. Calculate errors and probability-weighted squared errors.
3. Verify MSE=variance+bias².

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Estimate | Probability | Estimate × p | Error | Squared error | Squared error × p |
| --- | --- | --- | --- | --- | --- |
| 0 | 0.25 | 0 | -2 | 4 | 1 |
| 2 | 0.5 | 1 | 0 | 0 | 0 |
| 4 | 0.25 | 1 | 2 | 4 | 1 |

| Property | Value |
| --- | --- |
| Expectation | 2 |
| Bias | 0 |
| Variance | 2 |
| MSE | 2 |
| Variance + bias² | 2 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


theta=2; values=[0,2,4]; probs=[.25,.5,.25]; expectation=sum(v*p for v,p in zip(values,probs))
rows=[[v,p,v*p,v-theta,(v-theta)**2,p*(v-theta)**2] for v,p in zip(values,probs)]
show_table(["Estimate","Probability","Estimate × p","Error","Squared error","Squared error × p"],rows)
bias=expectation-theta; variance=sum(p*(v-expectation)**2 for v,p in zip(values,probs)); mse=sum(r[-1] for r in rows)
show_table(["Property","Value"],[["Expectation",expectation],["Bias",bias],["Variance",variance],["MSE",mse],["Variance + bias²",variance+bias*bias]])
assert abs(mse-variance-bias*bias)<1e-12


### Decision and interpretation

This estimator is unbiased in the stated sampling distribution, but estimates still vary. Unbiasedness does not mean every estimate equals the parameter.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
pop=rng.lognormal(1.5,.7,100000); theta=pop.mean(); fig,ax=plt.subplots(1,3,figsize=(10,3))
for a,n in zip(ax,[5,30,200]):
 z=[rng.choice(pop,n).mean() for _ in range(1000)]; a.hist(z,bins=25,density=True); a.axvline(theta,color='red'); a.set_title('n='+str(n))
plt.show()
true=pop.var(); v0=[]; v1=[]
for _ in range(1000):
 s=rng.choice(pop,12); v0.append(s.var(ddof=0)); v1.append(s.var(ddof=1))
print('true var',round(true,2),'average ddof0',round(np.mean(v0),2),'ddof1',round(np.mean(v1),2))

### Practice set · Sampling distributions
1. If σ=15 and n=100, find SE(x̄). What happens if n becomes 400?
2. An estimator has variance 4 and bias 1. Find its MSE.
3. Why can a skewed population still produce an approximately normal sample-mean distribution for sufficiently large n?
4. Explain the 95% confidence statement using repeated samples.

<details><summary>Answer sketches</summary>
1. 1.5; at n=400, .75. 2. 4+1²=5. 3. CLT conditions; finite variance and adequate n, with adequacy depending on skew/tails. 4. In repeated use, about 95% of intervals cover the fixed parameter when assumptions hold.
</details>

### Debrief
Distinguish bias from standard error. Which tends to shrink as n grows?

Record the formula, assumptions, and one follow-up question.